<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Jason_Lim_hw_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class: LLM Fundamentals - Class Lab 1b - OpenAI LLM API
# Topic: How an LLM generates the next token?

# Jason Lim - HW 1b
## Experiment 1: Different prompts
- France → top token: Paris (very confident)
- Dog sat on prompt
- Fox prompt
## Experiment 2: top_logprobs
- 1 vs 20 showed more/less alternative words
## Experiment 3: temperature
- 0 = more predictable
- 1.5 = more random

In [3]:
# OPENAI_API_KEY

In [4]:
print("Hello, LLM Fundamentals class ....")

Hello, LLM Fundamentals class ....


In [2]:
import math
import os, requests

# Sample code to show converting raw numbers to log and inverse-log

In [1]:
import math

original_prob = 0.7
logprob = math.log(original_prob)
prob = math.exp(logprob)

print(f"Original probability: {original_prob}")
print(f"Log probability: {logprob}")
print(f"Reconstructed probability: {prob}")

Original probability: 0.7
Log probability: -0.35667494393873245
Reconstructed probability: 0.7


# Create your own OpenAI API Key

In [5]:
from google.colab import userdata
# userdata.get('OPENAI_API_KEY')



---



In [6]:
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
text_to_embed = "How are you?"
# text_to_embed = "The quick brown fox jumps over the"
# text_to_embed = "The dog sat on"
# text_to_embed = "I have a dream"

response = client.embeddings.create(
    input=text_to_embed,
    model="text-embedding-3-small"
)

embedding_vector = response.data[0].embedding

print(f"Embedding for the text: {embedding_vector}")
print(f"Length of the embedding vector: {len(embedding_vector)}")

Embedding for the text: [0.027587890625, -0.0265655517578125, -0.027069091796875, 0.05780029296875, -0.00347900390625, -0.0028285980224609375, 0.0318603515625, 0.01013946533203125, -0.03411865234375, -0.044677734375, -0.034515380859375, -0.00939178466796875, -0.0223388671875, -0.0004012584686279297, -0.0132598876953125, 0.0213470458984375, -0.0285186767578125, 0.01468658447265625, 0.0079498291015625, 0.004047393798828125, 0.0257110595703125, -0.01910400390625, 0.0015573501586914062, 0.0227813720703125, 0.05047607421875, -0.0004570484161376953, 0.006866455078125, 0.0467529296875, -0.01085662841796875, -0.02374267578125, 0.04266357421875, -0.0396728515625, 0.0253448486328125, 0.035308837890625, -0.0070648193359375, 0.040374755859375, -0.05706787109375, 0.0205078125, -0.00421142578125, -0.06329345703125, -0.002712249755859375, -0.047515869140625, 0.010528564453125, 0.01959228515625, 0.006191253662109375, 0.0184478759765625, -0.034759521484375, 0.0008993148803710938, 0.032745361328125, 0.0

# "text-embedding-3-small" has default dimensions=1526, it can be changed to 512. By using the dimensions parameter, you can control the trade-off between embedding size and the model's ability to capture the nuances of the text.

# Embeddings are numerical representations (vectors) of text that capture its semantic meaning. To get them, you will make an API call to the embeddings endpoint, which returns a list of floating-point numbers representing the input text.

# OpenAI API call for text completions to generate the "next token" using gpt-4o

Reference:
https://platform.openai.com/docs/api-reference/chat


logprobs: Whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message.


top_logprobs: An integer between 0 and 20 specifying the number of most likely tokens to return at each token position, each with an associated log probability.

In [7]:
# api_key = os.environ.get("OPENAI_API_KEY")
api_key = userdata.get('OPENAI_API_KEY')

In [8]:
# print(api_key)

# Using gpt-4o to generate text completion response

# Notice the usage of "logprobs" and "top_logprobs"

https://developers.openai.com/cookbook/examples/using_logprobs

In [26]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={
        "Authorization":  f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "gpt-4o",
        "messages": [
            {"role": "user", "content": "The quick brown fox jumps over the"}
        ],
        "logprobs": True,
        "top_logprobs": 10,
        "temperature": 0,
        "max_tokens": 5
    }
)

# A successful execution of response should result in response of <Response [200]>

In [10]:
print(response)

<Response [200]>


# Convert the response to json, then extract words and probabilities

In [11]:
response_json = response.json()
logprobs = response_json["choices"][0]["logprobs"]
next_token_logprobs = logprobs["content"][0]["top_logprobs"]

# Check the contents of the dictionary

bytes are "vector representation" of the words and symbols

In [12]:
next_token_logprobs

[{'token': 'Paris',
  'logprob': -0.003832140937447548,
  'bytes': [80, 97, 114, 105, 115]},
 {'token': 'The', 'logprob': -5.5663323402404785, 'bytes': [84, 104, 101]},
 {'token': ' Paris',
  'logprob': -15.25383186340332,
  'bytes': [32, 80, 97, 114, 105, 115]},
 {'token': 'the', 'logprob': -16.05070686340332, 'bytes': [116, 104, 101]},
 {'token': 'Par', 'logprob': -18.95695686340332, 'bytes': [80, 97, 114]}]

# Take inverse-log to convert probabilities to normal numbers

In [13]:
token_prob = {}
for item in next_token_logprobs:
    token, logprob = item["token"], item["logprob"]
    prob = math.exp(logprob)
    print(token, prob)
    token_prob[token] = prob

Paris 0.9961751923442543
The 0.0038244816384737676
 Paris 2.3732582204187983e-07
the 1.069711287774693e-07
Par 5.849223828446404e-09


In [14]:
token_prob

{'Paris': 0.9961751923442543,
 'The': 0.0038244816384737676,
 ' Paris': 2.3732582204187983e-07,
 'the': 1.069711287774693e-07,
 'Par': 5.849223828446404e-09}

In [15]:
average_prob = sum(token_prob.values()) / len(token_prob)
print(f"Average probability: {average_prob}")

Average probability: 0.20000000482578054


# Apply "Greedy Sampling" (token with the highest probability to be the "Next Token")

In [16]:
# Find the key with the highest value
highest_value_key = max(token_prob, key=token_prob.get)

In [17]:
print(f"The key with the highest value is: {highest_value_key}")

The key with the highest value is: Paris


In [17]:
import openai

# Assuming 'client' is an initialized OpenAI client from previous cells.
# The question to ask the AI, specifically requesting it to use web search for current information.
user_question = "Please perform a web search to find the current teams that are remaining in the World Cup. List them in your response. Ensure the information is up-to-date."

# Using the OpenAI client to ask the question.
# The system message guides the AI to use its web search capabilities (if any) or knowledge base.
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to up-to-date information via web search capabilities. When asked for current events or real-time data, you are expected to use web search."},
    {"role": "user", "content": user_question}
]

print(f"Sending query to AI: \"{user_question}\"")

try:
    completion = client.chat.completions.create(
        model="gpt-4o",  # Using the specified model
        messages=messages
    )

    ai_response = completion.choices[0].message.content
    print("\nAI's response:")
    print(ai_response)

except openai.APIError as e:
    print(f"An OpenAI API error occurred: {e}")
    print("Please ensure the OpenAI API key is valid and the model 'gpt-4o' is accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    print("If the AI model does not have direct real-time web access, it will answer based on its training data.")

# 1) Remember GPU Savings: Runtime >> Disconnect and delete run time.
# 2) Check you do not have any active sessions on the VM.

In [18]:
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "Tell me a short, funny joke about birds."}
    ]
)

print("Bird joke:")
print(completion.choices[0].message.content)

Bird joke:
Why do seagulls fly over the ocean? Because if they flew over the bay, they'd be bagels!
